In [ ]:
import pandas as pd
df = pd.read_parquet('../data/model_data.parquet')

x_cols0 = [c for c in df.columns if c.startswith("x")]
cond_cols0 = [c for c in df.columns if c.startswith("con")]

# fill missing values in cond columns with forward fill
# backward fill is needed for the first rows if they are missing (that is lookfowarding but very few rows are missing in the beginning after forward fill)
df["cond2"] = df["cond2"].ffill()
df["cond2"] = df["cond2"].bfill()

df["cond3"] = df["cond3"].ffill()
df["cond3"] = df["cond3"].bfill()

In [ ]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Optional, Dict
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.cluster import KMeans


# --------------------------------------------------
# Helpers
# --------------------------------------------------

def get_year_index(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s).dt.to_period("Y")


def fit_clip_and_scale_cond(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cond_cols: List[str],
    q_low: float = 0.01,
    q_high: float = 0.99,
    final_clip: float = 5.0,
) -> tuple[np.ndarray, np.ndarray]:
    C_train = train_df[cond_cols].to_numpy(dtype=float)
    C_test = test_df[cond_cols].to_numpy(dtype=float)

    lower = np.quantile(C_train, q_low, axis=0)
    upper = np.quantile(C_train, q_high, axis=0)

    C_train = np.clip(C_train, lower, upper)
    C_test = np.clip(C_test, lower, upper)

    mean = C_train.mean(axis=0)
    std = C_train.std(axis=0)
    std = np.where(std < 1e-12, 1.0, std)

    C_train = (C_train - mean) / std
    C_test = (C_test - mean) / std

    C_train = np.clip(C_train, -final_clip, final_clip)
    C_test = np.clip(C_test, -final_clip, final_clip)

    return C_train, C_test



# --------------------------------------------------
# Window schedule
# --------------------------------------------------

@dataclass
class WindowSpec:
    mode: str              # "rolling" or "expanding"
    train_years: int = 4   # used for rolling
    min_train_years: int = 4


def build_yearly_schedule(
    df: pd.DataFrame,
    date_col: str,
    window_spec: WindowSpec,
) -> List[Dict]:
    tmp = df[[date_col]].copy()
    tmp[date_col] = pd.to_datetime(tmp[date_col])
    years = np.array(sorted(tmp[date_col].dt.to_period("Y").unique()))

    schedule = []
    for i in range(len(years) - 1):
        pred_year = years[i + 1]

        if window_spec.mode == "rolling":
            start_idx = i - window_spec.train_years + 1
            if start_idx < 0:
                continue
            train_years = years[start_idx:i + 1]

        elif window_spec.mode == "expanding":
            if i + 1 < window_spec.min_train_years:
                continue
            train_years = years[:i + 1]

        else:
            raise ValueError("window_spec.mode must be 'rolling' or 'expanding'")

        schedule.append({
            "train_years": train_years,
            "pred_year": pred_year,
        })

    return schedule


# --------------------------------------------------
# MoE
# --------------------------------------------------

class LogisticGateRidgeMoE:
    def __init__(
        self,
        n_experts: int = 3,
        ridge_alpha: float = 1.0,
        gate_C: float = 1.0,
        n_em_iters: int = 10,
        tau: Optional[float] = None,
        random_state: int = 0,
    ):
        self.n_experts = n_experts
        self.ridge_alpha = ridge_alpha
        self.gate_C = gate_C
        self.n_em_iters = n_em_iters
        self.tau = tau
        self.random_state = random_state

        self.experts = None
        self.gate = None

    def _init_models(self, X: np.ndarray, C: np.ndarray, y: np.ndarray):
        labels = KMeans(
            n_clusters=self.n_experts,
            n_init=20,
            random_state=self.random_state,
        ).fit_predict(C)

        self.experts = []
        for k in range(self.n_experts):
            mask = labels == k
            if mask.sum() < max(20, X.shape[1]):
                mask = np.ones(len(y), dtype=bool)

            model = Ridge(alpha=self.ridge_alpha, fit_intercept=False)
            model.fit(X[mask], y[mask])
            self.experts.append(model)

        self.gate = LogisticRegression(
            C=self.gate_C,
            multi_class="multinomial",
            solver="lbfgs",
            max_iter=1000,
            random_state=self.random_state,
        )
        self.gate.fit(C, labels)

    def _expert_preds(self, X: np.ndarray) -> np.ndarray:
        return np.column_stack([m.predict(X) for m in self.experts])

    def _responsibilities(
        self,
        y: np.ndarray,
        expert_preds: np.ndarray,
        gate_probs: np.ndarray,
    ) -> np.ndarray:
        sq_err = (y[:, None] - expert_preds) ** 2

        tau = self.tau
        if tau is None:
            tau = np.var(y)
            if tau < 1e-12:
                tau = 1.0

        likelihood = np.exp(-sq_err / tau)
        unnorm = gate_probs * likelihood
        denom = np.clip(unnorm.sum(axis=1, keepdims=True), 1e-12, None)
        return unnorm / denom

    def fit(self, X: np.ndarray, C: np.ndarray, y: np.ndarray):
        self._init_models(X, C, y)

        for _ in range(self.n_em_iters):
            expert_preds = self._expert_preds(X)
            gate_probs = self.gate.predict_proba(C)
            resp = self._responsibilities(y, expert_preds, gate_probs)

            new_experts = []
            for k in range(self.n_experts):
                m = Ridge(alpha=self.ridge_alpha, fit_intercept=False)
                m.fit(X, y, sample_weight=resp[:, k])
                new_experts.append(m)
            self.experts = new_experts

            labels = resp.argmax(axis=1)
            self.gate = LogisticRegression(
                C=self.gate_C,
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=1000,
                random_state=self.random_state,
            )
            self.gate.fit(C, labels)

        return self

    def predict_components(self, X: np.ndarray, C: np.ndarray):
        expert_preds = self._expert_preds(X)
        gate_probs = self.gate.predict_proba(C)
        pred = (expert_preds * gate_probs).sum(axis=1)
        return pred, expert_preds, gate_probs


# --------------------------------------------------
# Yearly walk-forward
# --------------------------------------------------

def walk_forward_yearly_predictions(
    df: pd.DataFrame,
    x_cols: List[str] = x_cols0,
    cond_cols: List[str] = cond_cols0,
    target_col: str = "ret_fopen",
    date_col: str = "msgStamp",
    trade_date_col: str = "trade_date",
    window_spec: WindowSpec = WindowSpec(mode="expanding", train_years=4),
    model_type: str = "moe",   # "moe" or "ridge"
    ridge_alpha: float = 1.0,
    gate_C: float = 1.0,
    n_experts: int = 3,
    n_em_iters: int = 3,
    tau: Optional[float] = None,
    x_clip: Optional[float] = 3.0,
    random_state: int = 0,
    y_clip_quantile: Optional[float] = 0.01,
) -> pd.DataFrame:

    data = df.copy()
    data[date_col] = pd.to_datetime(data[date_col])
    data = data.sort_values(date_col).reset_index(drop=True)
    data["year"] = data[date_col].dt.to_period("Y")

    if trade_date_col not in data.columns:
        data[trade_date_col] = data[date_col].dt.date

    schedule = build_yearly_schedule(data, date_col, window_spec)
    outputs = []

    for step in schedule:
        train_years = step["train_years"]
        pred_year = step["pred_year"]

        train_df = data.loc[data["year"].isin(train_years)].copy()
        test_df = data.loc[data["year"] == pred_year].copy()

        train_df = train_df.dropna(subset=x_cols + cond_cols + [target_col]).copy()
        test_df = test_df.dropna(subset=x_cols + cond_cols).copy()

        if len(train_df) == 0 or len(test_df) == 0:
            continue

        X_train = train_df[x_cols].to_numpy(dtype=float)
        X_test = test_df[x_cols].to_numpy(dtype=float)
        y_train = train_df[target_col].to_numpy(dtype=float)

        if y_clip_quantile is not None:
            low = np.quantile(y_train, y_clip_quantile)
            high = np.quantile(y_train, 1 - y_clip_quantile)
            y_train = np.clip(y_train, low, high)

        if x_clip is not None:
            X_train = np.clip(X_train, -x_clip, x_clip)
            X_test = np.clip(X_test, -x_clip, x_clip)

        C_train, C_test = fit_clip_and_scale_cond(train_df, test_df, cond_cols)

        if model_type == "ridge":
            model = Ridge(alpha=ridge_alpha*len(train_df), fit_intercept=False)
            model.fit(X_train, y_train)
            coef_abs_sum = np.abs(model.coef_).sum()
            pred_test = model.predict(X_test)
            if coef_abs_sum > 0:
                pred_test = pred_test / coef_abs_sum

            out = test_df[[date_col, trade_date_col]].copy()
            out["pred"] = pred_test

        elif model_type == "moe":
            model = LogisticGateRidgeMoE(
                n_experts=n_experts,
                ridge_alpha=ridge_alpha,
                gate_C=gate_C,
                n_em_iters=n_em_iters,
                tau=tau,
                random_state=random_state,
            )
            model.fit(X_train, C_train, y_train)
            pred_test, expert_preds_test, gate_probs_test = model.predict_components(X_test, C_test)

            out = test_df[[date_col, trade_date_col]].copy()
            out["pred"] = pred_test

            for k in range(n_experts):
                out[f"expert_{k}"] = expert_preds_test[:, k]
                out[f"gate_{k}"] = gate_probs_test[:, k]

        else:
            raise ValueError("model_type must be 'ridge' or 'moe'")

        out["pred_year"] = str(pred_year)
        out["train_start_year"] = str(train_years[0])
        out["train_end_year"] = str(train_years[-1])
        outputs.append(out)

    if not outputs:
        return pd.DataFrame(columns=[date_col, trade_date_col, "pred"])

    return pd.concat(outputs, axis=0).sort_values(date_col).reset_index(drop=True)

In [ ]:
ridge_alphas = [0, 1e-6, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 1e1, 1e2, 1e3, 1e4] 

preds_exp_by_alpha = {}
preds_exp_long = []
preds_exp_wide = []

for alpha in ridge_alphas:
    preds = walk_forward_yearly_predictions(
        df=df,
        x_cols=x_cols0,
        model_type="ridge",
        ridge_alpha=alpha,
        x_clip=3.0,
        random_state=42,
    )

    preds_exp_by_alpha[alpha] = preds.copy()
    preds_exp_long.append(preds.assign(ridge_alpha=alpha))
    preds_exp_wide.append(
        preds[["msgStamp", "trade_date", "pred"]]
        .rename(columns={"pred": f"pred_alpha_{alpha:g}"})
        .set_index(["msgStamp", "trade_date"])
    )

preds_exp_all_long = pd.concat(preds_exp_long, ignore_index=True)
preds_exp_all_wide = pd.concat(preds_exp_wide, axis=1).reset_index()



In [ ]:
preds_exp_all_wide.to_parquet('global_reg.parquet')

In [ ]:
preds_exp_all_wide =pd.read_parquet('global_reg.parquet') 

In [ ]:
preds_exp_all_wide = preds_exp_all_wide.rename(
    columns=lambda c: f"a_{c.replace('pred_alpha_', '', 1)}" if c.startswith("pred_alpha_") else c
)


In [ ]:
pred_cols = [c for c in preds_exp_all_wide.columns if c.startswith("a_")]

In [ ]:
# add the returns to the predictions table
dd = preds_exp_all_wide.merge(df[["msgStamp", "ret_fopen"]], on="msgStamp", how="left")

In [ ]:
import datetime
def backtest(df, feature_col, ret_col, date_col="trade_date", annualization=252):
    data = df[[date_col, feature_col, ret_col]].dropna().copy()
    data[date_col] = pd.to_datetime(data[date_col]).dt.date

    def daily_stats(group):
        alphas = group[feature_col].clip(-3, 3)
        returns = group[ret_col]
        total_pnl = np.sum(alphas * returns)
        total_gross = np.sum(np.abs(alphas))
        return pd.Series({"daily_pnl": total_pnl, "daily_gross": total_gross})

    daily_stats_df = (
        data.groupby(date_col, sort=True)
        .apply(daily_stats)
        .dropna()
    )

    daily_pnl = daily_stats_df["daily_pnl"]
    daily_gross = daily_stats_df["daily_gross"]

    cumulative_returns = daily_pnl.cumsum()
    sharpe_ratio = np.sqrt(annualization) * daily_pnl.mean() / daily_pnl.std()
    average_return = daily_pnl.sum()/daily_gross.sum() if daily_gross.sum() > 0 else 0.0

    return {
        "cumulative_returns": cumulative_returns,
        "sharpe_ratio": sharpe_ratio,
        "average_return": average_return,
    }

bt= {}
for pr in pred_cols:
    bt[pr] = backtest(
        dd[(dd["trade_date"] < datetime.date(2023, 1, 1))],
        feature_col=pr,
        ret_col="ret_fopen",
    )


In [ ]:
metrics = pd.DataFrame(
    {
        "alpha": pred_cols,
        "sharpe_ratio": [bt[col]["sharpe_ratio"] for col in pred_cols],
        "return": [bt[col]["average_return"] for col in pred_cols],
    }
)
metrics["alpha_value"] = metrics["alpha"].str.replace("a_", "").astype(float)
metrics["alpha"] = metrics["alpha_value"].map(lambda x: f"{x:.2e}")
metrics = metrics.sort_values("alpha_value").drop(columns="alpha_value")
metrics.set_index("alpha").T


In [ ]:
cum_pnl = pd.concat(
    [bt[col]["cumulative_returns"].rename(col) for col in pred_cols],
    axis=1,
)
ax = cum_pnl.plot(figsize=(12, 6), title="Cumulative PnL by Alpha")
ax.set_ylabel("Cumulative PnL")
ax.legend(title="alpha", bbox_to_anchor=(1.02, 1), loc="upper left")
